In [57]:
import pandas as pd
import numpy as np


data = [100, 102, 101, 103, 105, 104, 106, 108, 107, 109,
111, 110, 108, 106, 104, 103, 101, 100, 98, 99,
101, 103, 102, 104, 106, 108, 110, 109, 111, 113,
112, 110, 108, 107, 105, 103, 104, 106, 108, 107,
109, 111, 113, 115, 114, 116, 118, 117, 115, 113,
112, 110, 108, 109, 111, 113, 112, 114, 116, 118,
117, 119, 121, 120, 122, 124, 123, 121, 119, 120,
122, 124, 126, 125, 127, 129, 128, 130, 132, 131]

fast_window = 5
slow_window = 10

df = pd.DataFrame({'Prices' : data})
df['Return'] = df['Prices'].pct_change()
df['Fast_MA'] = df['Prices'].rolling(window=fast_window).mean()
df['Slow_MA'] = df['Prices'].rolling(window=slow_window).mean()
df['Signal'] = np.where(df['Fast_MA'] > df['Slow_MA'],1,0)
df['Strategy_Return'] = df['Signal'].shift(1) * df['Return']
df['Strategy_Return']= df['Strategy_Return'].fillna(0)
df['Equity_Curve'] = (1 + df['Strategy_Return']).cumprod()
df

# Entry_Index Exit_Index Entry_Price Exit_Price Trade_Return Holding_Period Win/Loss 

entry_index = df.index[df['Signal'] > df['Signal'].shift(1)].tolist()
exit_index = df.index[df['Signal']< df['Signal'].shift(1)].tolist()

entry_index = entry_index[:len(exit_index)]
entry_price = df.loc[entry_index,'Prices'].tolist()
exit_price = df.loc[exit_index,'Prices'].tolist()

trade_data = {
    'Entry Index' : entry_index,
    'Exit Index' : exit_index,
    'Entry Price' : entry_price,
    'Exit Price' : exit_price
}

df1 = pd.DataFrame(trade_data)
df1['Trade Return'] = (df1['Exit Price'] - df1['Entry Price'])/df1['Entry Price']
df1['Holding Period'] = df1['Exit Index'] - df1['Entry Index']
df1['Win/Loss'] = np.where(df1['Trade Return'] > 0,'Win','Loss')

best_trade = df1.loc[df1['Trade Return'].idxmax()]
worst_trade = df1.loc[df1['Trade Return'].idxmin()]

average_return = df1['Trade Return'].mean()
median_return = df1['Trade Return'].median()

average_holding_period = df1['Holding Period'].mean()
max_holding_period = df1['Holding Period'].max()
min_holding_period = df1['Holding Period'].min()

current_win_streak = 0
max_win_streak = 0

current_loss_streak =0
max_loss_streak = 0

for result in df1['Win/Loss'] :
    if result == 'Win':
        current_win_streak +=1
        max_win_streak = max(current_win_streak,max_win_streak)

        current_loss_streak = 0
    else :
        current_loss_streak +=1
        max_loss_streak = max(current_loss_streak,max_loss_streak)
        current_win_streak = 0


winning_trades = (df1['Trade Return'] > 0).sum()
losing_trades = (df1['Trade Return'] < 0).sum()
breakeven_trades = (df1['Trade Return'] == 0).sum()
total_trades = len(df1)

win_rate = winning_trades/total_trades
loss_rate = losing_trades/total_trades

df1['Cummulative Trade Return'] = (1 + df1['Trade Return']).cumprod()

summary = {
    'Total Trades': total_trades,
    'Winning Trades': winning_trades,
    'Losing Trades': losing_trades,
    'Breakeven Trades': breakeven_trades,
    'Win Rate %': win_rate * 100,
    'Loss Rate %': loss_rate * 100,
    'Best Trade Return': best_trade['Trade Return'],
    'Worst Trade Return': worst_trade['Trade Return'],
    'Average Trade Return': average_return,
    'Median Trade Return': median_return,
    'Average Holding Period': average_holding_period,
    'Max Holding Period': max_holding_period,
    'Min Holding Period': min_holding_period,
    'Max Consecutive Wins': max_win_streak,
    'Max Consecutive Losses': max_loss_streak
}

summary_df = pd.DataFrame([summary])
summary_df







,Total Trades,Winning Trades,Losing Trades,Breakeven Trades,Win Rate %,Loss Rate %,Best Trade Return,Worst Trade Return,Average Trade Return,Median Trade Return,Average Holding Period,Max Holding Period,Min Holding Period,Max Consecutive Wins,Max Consecutive Losses
0,4,3,1,0,75.0,25.0,0.070175,-0.055046,0.00848,0.009395,10.25,13,6,3,1
